In [1]:
import asyncio
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_ENDPOINT = "http://localhost:8000/mcp"

client = MultiServerMCPClient({
    "hr": {
        "transport": "http",
        "url": MCP_ENDPOINT
    }
})

# Get all MCP Server Tools
cached_tools = await client.get_tools()
print(f"\nLoaded {len(cached_tools)} tools and cached.\n")

# Build Skill Map for all MCP Server Tools
def build_skill_map(tools):
    skill_map = {}

    for t in tools:
        metadata = getattr(t, "metadata", None) or {}
        skill = metadata.get("skill", "unknown")
        
        args_schema = getattr(t, "args_schema", None) or {}

        tool_spec = {
            "name": t.name,
            "description": t.description.strip(),
            "parameters": args_schema
        }

        skill_map.setdefault(skill, []).append(tool_spec)

    return skill_map

# Select Tools based on selected Skills
def build_selected_stools(selected_skills, cached_skill_map, cached_valid_skills):
    selected_tools = []
    
    for skill in selected_skills:
        if skill not in cached_valid_skills:
            #raise ValueError(f"Unknown skill: {skill}")
            print(f">>>>> Warning: Unknown skill: {skill} in my cached skills!")
        selected_tools.extend(
            cached_skill_map.get(skill, [])
        )
        
    return selected_tools


# --- Test ---
cached_skill_map = build_skill_map(cached_tools)
cached_valid_skills = set(cached_skill_map.keys())
print("--- Full Skill Map ---")
print(json.dumps(cached_skill_map, indent=2))
print("===================")

selected_skills = ['policy','leave','employee_profile','bad']

# FIXED: Passed skill_map explicitly as an argument
selected_tools = build_selected_stools(selected_skills, cached_skill_map, cached_valid_skills)

print("--- Selected Tools ---")
print(json.dumps(selected_tools, indent=2))


Loaded 14 tools and cached.

--- Full Skill Map ---
{
  "employee_profile": [
    {
      "name": "get_employee_basic_profile",
      "description": "ROLE: ANY EMPLOYEE\n\nGet basic employee basic profile information using employee_code",
      "parameters": {
        "additionalProperties": false,
        "properties": {
          "employee_code": {
            "type": "string"
          }
        },
        "required": [
          "employee_code"
        ],
        "type": "object"
      }
    },
    {
      "name": "get_employee_detailed_profile",
      "description": "ROLE: ANY EMPLOYEE (self lookup) OR HR/ADMIN\n\nGet full employee profile information including contacts, employment, balance, compensation.",
      "parameters": {
        "additionalProperties": false,
        "properties": {
          "employee_code": {
            "type": "string"
          }
        },
        "required": [
          "employee_code"
        ],
        "type": "object"
      }
    },
    {
      